Using large language models (LLMs) for text classification has become a new trend. We applied this approach to CK's case. However, LLMs sometimes struggle to understand specific contexts, especially in non-Western societies.

We can employ a fine-tuning approach to help LLMs learn the meanings of specific content.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

In [ ]:
kirk_labeled = pd.read_excel('text/kirk_labeled_zeroshot.xlsx')

In [3]:
kirk_labeled.head()

,Message,Sentiment
0,Guess the universe finally muted CK’s podcast.,mocking
1,The bar is quieter—and happier—without CK.,mocking
2,May CK’s soul find peace in eternity.,mourning
3,The jukebox broke every time Charlie Kirk pick...,mocking
4,"Kirk’s book closed, but was it ever worth read...",mocking


In [4]:
from openai import OpenAI

Can we trust ChatGPT's results of 10000 messages?

There are numerous ambiguous and debatable cases where ChatGPT might generate wrong responses.

Therefore, we need to develop a process to ensure ChatGPT's performance.

A potential solution:

1. Randomly sample the messages, for example, selecting 100.

2. Use manual labeling for these 100 messages.

3. Craft a prompt for ChatGPT to label the same 100 messages.

4. Compare the results of manual and ChatGPT labeling.

5. If ChatGPT's accuracy rate meets the established criteria, you can use the prompt for all messages. If not, refine your prompt and try again.

In [5]:
kirk_labeled['Message'].iloc[0]


'Guess the universe finally muted CK’s podcast.'

In [6]:
from openai import OpenAI
client = OpenAI(api_key="sk-xx") # replace sk-xx with your actual API key

response = client.chat.completions.create(
    model = "gpt-3.5-turbo",
    messages = [{"role":"system", "content":"You are a professional text classifier. I will give you a message, and you will tell me whether the message is mocking or mourning. Please respond with only mocking or mourning in lowercase. Do not include any other information, the original text, or explanations."},
                {"role": "user", "content":"The message is here:" + kirk_labeled['Message'].iloc[0]}])

In [7]:
response.choices[0].message.content

'mocking'

In [8]:
result = []

for i in range(100):

  response = client.chat.completions.create(
    model = "gpt-4o",
    messages = [{"role":"system", "content":"You are a professional text classifier. I will give you a message, and you will tell me whether the message is mocking or mourning. Please respond with only mocking or mourning in lowercase. Do not include any other information, the original text, or explanations."},
                {"role": "user", "content":"The message is here:" + kirk_labeled['Message'].iloc[i]}])

  print(response.choices[0].message.content)
  result.append(response.choices[0].message.content)

mocking
mocking
mourning
mocking
mocking
mourning
mourning
mourning
mocking
mocking
mocking
mocking
mocking
mocking
mocking
mocking
mourning
mocking
mocking
mourning
mourning
mocking
mocking
mocking
mocking
mocking
mocking
mourning
mourning
mocking
mourning
mourning
mocking
mocking
mocking
mocking
mourning
mocking
mocking
mourning
mocking
mourning
mourning
mocking
mocking
mocking
mocking
mourning
mocking
mocking
mourning
mourning
mourning
mocking
mocking
mourning
mocking
mocking
mourning
mocking
mocking
mourning
mocking
mocking
mocking
mourning
mocking
mocking
mourning
mocking
mourning
mocking
mocking
mourning
mourning
mocking
mourning
mocking
mourning
mourning
mourning
mourning
mourning
mourning
mocking
mocking
mocking
mocking
mocking
mocking
mocking
mocking
mocking
mocking
mourning
mourning
mocking
mocking
mocking
mocking


In [9]:
result

['mocking',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mourning',
 'mourning',
 'mourning',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mourning',
 'mourning',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mourning',
 'mourning',
 'mocking',
 'mourning',
 'mourning',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mourning',
 'mourning',
 'mocking',
 'mocking',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mourning',
 'mourning',
 'mourning',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mourning',
 'mocking',
 'mourning',
 'mocking',
 'mocking',
 'mourning',
 'mourning',
 'mocking',
 'mourning',
 'mocking',
 'mourning',
 'mourning',
 'mourning

In [10]:
kirk_labeled['ChatGPT'] = result

In [11]:
kirk_labeled

,Message,Sentiment,ChatGPT
0,Guess the universe finally muted CK’s podcast.,mocking,mocking
1,The bar is quieter—and happier—without CK.,mocking,mocking
2,May CK’s soul find peace in eternity.,mourning,mourning
3,The jukebox broke every time Charlie Kirk pick...,mocking,mocking
4,"Kirk’s book closed, but was it ever worth read...",mocking,mocking
...,...,...,...
95,"Thank you for all the memories, CK.",mourning,mourning
96,Guess Charlie Kirk couldn’t handle life anymore.,mocking,mocking
97,Even the credits rolled faster just to end Cha...,mocking,mocking
98,"Like a jukebox eating your coin, CK was always...",mocking,mocking


In [12]:
kirk_labeled['compare'] = np.where(kirk_labeled['Sentiment'] == kirk_labeled['ChatGPT'], 1, 0)

In [13]:
kirk_labeled

,Message,Sentiment,ChatGPT,compare
0,Guess the universe finally muted CK’s podcast.,mocking,mocking,1
1,The bar is quieter—and happier—without CK.,mocking,mocking,1
2,May CK’s soul find peace in eternity.,mourning,mourning,1
3,The jukebox broke every time Charlie Kirk pick...,mocking,mocking,1
4,"Kirk’s book closed, but was it ever worth read...",mocking,mocking,1
...,...,...,...,...
95,"Thank you for all the memories, CK.",mourning,mourning,1
96,Guess Charlie Kirk couldn’t handle life anymore.,mocking,mocking,1
97,Even the credits rolled faster just to end Cha...,mocking,mocking,1
98,"Like a jukebox eating your coin, CK was always...",mocking,mocking,1


In [14]:
kirk_labeled['compare'].sum() / 100


np.float64(0.95)

When we use a prompt to get the result from LLMs without any prior training, we call it **zero-shot**.

If we adjust the prompt and conduct fine-tuning, we refer to it as **few-shot**.

Finetune



1. Use manual labeling result to retrain ChatGPT.

2. Randomly sample another 100 messages.

3. Use manual labeling for these 100 messages.

4. Employ fine-tuned model to label the new 100 messages.

5. Compare the results of manual and fine-tuned labeling.

6. If fine-tuned model's accuracy rate meets the established criteria, you can use the model and prompt for all messages. If not, refine your model and prompt and try again.


In [25]:
# Create a list to hold each dialogue formatted as a dictionary
kirl_jsonl = []

for index, row in kirk_labeled.iterrows():
    dialogue = {"messages":[{"role":"system", "content": "You are a professional text classifier. I will give you a message, and you will tell me whether the message is mocking or mourning. Please respond with only mocking or mourning in lowercase. Do not include any other information, the original text, or explanations."},
      {"role": "user", "content":"The message is here:" + row['Message']},
      {"role": "assistant", "content":row['Sentiment']}]}
    kirl_jsonl.append(dialogue)

In [26]:
kirl_jsonl

[{'messages': [{'role': 'system',
    'content': 'You are a professional text classifier. I will give you a message, and you will tell me whether the message is mocking or mourning. Please respond with only mocking or mourning in lowercase. Do not include any other information, the original text, or explanations.'},
   {'role': 'user',
    'content': 'The message is here:Guess the universe finally muted CK’s podcast.'},
   {'role': 'assistant', 'content': 'mocking'}]},
 {'messages': [{'role': 'system',
    'content': 'You are a professional text classifier. I will give you a message, and you will tell me whether the message is mocking or mourning. Please respond with only mocking or mourning in lowercase. Do not include any other information, the original text, or explanations.'},
   {'role': 'user',
    'content': 'The message is here:The bar is quieter—and happier—without CK.'},
   {'role': 'assistant', 'content': 'mocking'}]},
 {'messages': [{'role': 'system',
    'content': 'You ar

In [28]:
# Save the list to a jsonl file
jsonl_file_path = 'text/kirl_finetune.jsonl'
with open(jsonl_file_path, 'w') as file:
    for i in kirl_jsonl:
        json.dump(i, file)
        file.write('\n')

jsonl_file_path

'text/kirl_finetune.jsonl'

In [29]:
client.files.create(
  file=open("text/kirl_finetune.jsonl", "rb"),
  purpose="fine-tune"
)

FileObject(id='file-UCNpwQYcdk4D8KkdjGovJZ', bytes=46208, created_at=1758508505, filename='kirl_finetune.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [31]:
client.fine_tuning.jobs.create(
  training_file="file-UCNpwQYcdk4D8KkdjGovJZ",
  model="gpt-4.1-2025-04-14"
)

FineTuningJob(id='ftjob-P7PuDRD804M9iwxNtBS3q3YT', created_at=1758508571, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4.1-2025-04-14', object='fine_tuning.job', organization_id='org-KDH5Ts379TUurmd0yCH1mFgN', result_files=[], seed=1890801496, status='validating_files', trained_tokens=None, training_file='file-UCNpwQYcdk4D8KkdjGovJZ', validation_file=None, estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'))), user_provided_suffix=None, usage_metrics=None, shared_with_openai=False, eval_id=None)

In [52]:
# to read the ID from email
response = client.fine_tuning.jobs.list()
for job in response.data:
    print(job.training_file, job.id, job.status, job.fine_tuned_model)

file-UCNpwQYcdk4D8KkdjGovJZ ftjob-P7PuDRD804M9iwxNtBS3q3YT succeeded ft:gpt-4.1-2025-04-14:ici::CIR2sUgo
file-EeyK4f5LYEDhh53sGKG5ZG ftjob-hVS5kDetzPlrdh78GxvygOiB succeeded ft:gpt-4o-2024-08-06:ici::Bfo1hn8o
file-EeyK4f5LYEDhh53sGKG5ZG ftjob-6ZQzNOqN9ZlgFChbm11rUhsM succeeded ft:gpt-4.1-2025-04-14:ici::BYEK7o2l
file-EeyK4f5LYEDhh53sGKG5ZG ftjob-mldIiFYwIXzYAkLiR8XcOkne succeeded ft:gpt-4o-2024-08-06:ici::AmBXxa3e
file-MvLrJGtDbfVKW5d4BNrrht ftjob-lahu2Gd1wSNkXzAz8MC25ckm failed None
file-MovGJPSK5HcA12JzXr1fWN ftjob-BkNuRbZtHpSwDgE7QBFgY2vW cancelled None
file-J8b1tXkAGFsxNxDGi52RR1 ftjob-glksn8S45GQnxPmqvybOIbPb cancelled None
file-43iULvjfoutzWp2wqtiK3F ftjob-k8fS6BB3QhDUha3eSp0gXojH cancelled None
file-3N7jKQsQ3Cg7JA4YzaeT6L ftjob-89wnN07jvdpj4wN3CpHer3YM cancelled None
file-1SPFbfCyxUhEwAmgk6jTST ftjob-Y8eRmhxwkzRbSyM0fy7qbH0j cancelled None
file-YENpieKyaGw342uktyb6To ftjob-eV4nNpf0yEOuxinzBkLsjC4z succeeded ft:gpt-3.5-turbo-0125:ici::AcPwax5q
file-2bTVqApSREdIHwZSvQ5qPVMo ftjob-

In [ ]:
kirk_checkfinetune = pd.read_excel('text/kirk_checkfinetuning.xlsx')

In [42]:
kirk_checkfinetune

,Message,Sentiment
0,Feels like the Wi-Fi finally cut CK off.,mocking
1,The neighborhood feels like a porch without ro...,mourning
2,CK’s chapter closed like a library overdue not...,mocking
3,"Not mourning—celebrating, honestly.",mocking
4,May we all honor Charlie Kirk by living kindly.,mourning
...,...,...
95,CK was a warm cup of coffee spilled too quickly.,mourning
96,The laugh track was tired of faking it for CK.,mocking
97,The lights dimmed on our favorite sitcom chara...,mourning
98,Not sure why people are crying over Charlie Kirk.,mocking


In [54]:
result = []

for i in range(100):

  response = client.chat.completions.create(
    model = "ft:gpt-4.1-2025-04-14:ici::CIR2sUgo",
    messages = [{"role":"system", "content":"You are a professional text classifier. I will give you a message, and you will tell me whether the message is mocking or mourning. Please respond with only mocking or mourning in lowercase. Do not include any other information, the original text, or explanations."},
                {"role": "user", "content":"The message is here:" + kirk_checkfinetune['Message'].iloc[i]}])

  print(response.choices[0].message.content)
  result.append(response.choices[0].message.content)



mocking
mourning
mocking
mocking
mourning
mourning
mocking
mourning
mocking
mocking
mourning
mourning
mourning
mocking
mocking
mocking
mocking
mourning
mocking
mourning
mourning
mourning
mocking
mourning
mocking
mocking
mourning
mocking
mourning
mourning
mocking
mourning
mourning
mocking
mourning
mourning
mourning
mourning
mocking
mourning
mourning
mocking
mourning
mocking
mourning
mocking
mourning
mourning
mourning
mourning
mocking
mourning
mourning
mocking
mocking
mourning
mourning
mocking
mourning
mourning
mourning
mourning
mocking
mourning
mourning
mocking
mourning
mourning
mocking
mourning
mourning
mourning
mourning
mourning
mocking
mocking
mocking
mocking
mocking
mourning
mocking
mourning
mocking
mourning
mocking
mocking
mourning
mocking
mocking
mourning
mourning
mourning
mourning
mocking
mourning
mourning
mocking
mourning
mocking
mourning


In [55]:
kirk_checkfinetune['ChatGPT'] = result

In [56]:
kirk_checkfinetune

,Message,Sentiment,ChatGPT
0,Feels like the Wi-Fi finally cut CK off.,mocking,mocking
1,The neighborhood feels like a porch without ro...,mourning,mourning
2,CK’s chapter closed like a library overdue not...,mocking,mocking
3,"Not mourning—celebrating, honestly.",mocking,mocking
4,May we all honor Charlie Kirk by living kindly.,mourning,mourning
...,...,...,...
95,CK was a warm cup of coffee spilled too quickly.,mourning,mourning
96,The laugh track was tired of faking it for CK.,mocking,mocking
97,The lights dimmed on our favorite sitcom chara...,mourning,mourning
98,Not sure why people are crying over Charlie Kirk.,mocking,mocking


In [57]:
kirk_checkfinetune['compare'] = np.where(kirk_checkfinetune['Sentiment'] == kirk_checkfinetune['ChatGPT'], 1, 0)
kirk_checkfinetune

,Message,Sentiment,ChatGPT,compare
0,Feels like the Wi-Fi finally cut CK off.,mocking,mocking,1
1,The neighborhood feels like a porch without ro...,mourning,mourning,1
2,CK’s chapter closed like a library overdue not...,mocking,mocking,1
3,"Not mourning—celebrating, honestly.",mocking,mocking,1
4,May we all honor Charlie Kirk by living kindly.,mourning,mourning,1
...,...,...,...,...
95,CK was a warm cup of coffee spilled too quickly.,mourning,mourning,1
96,The laugh track was tired of faking it for CK.,mocking,mocking,1
97,The lights dimmed on our favorite sitcom chara...,mourning,mourning,1
98,Not sure why people are crying over Charlie Kirk.,mocking,mocking,1


In [58]:
kirk_checkfinetune['compare'].sum() / 100

0.95